In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

config_path = "configs/central-kalimantan.yaml"

from palmdef_risk.io.run import create_or_resume_run
ctx = create_or_resume_run(config_path, resume=True)

Resuming existing run: wri_central-kalimantan_baseline_20260521_112147


In [3]:
import forestatrisk as far

ctx.output_dir.mkdir(parents=True, exist_ok=True)
far.sample(
    nsamp=ctx.config.nsamp,
    adapt=False,
    seed=ctx.config.seed,
    csize=ctx.config.csize,
    var_dir=str(ctx.data_dir),
    input_forest_raster="fcc23.tif",
    output_file=str(ctx.output_dir / "sample.csv"),
)

Sample 2x 100000 pixels (deforested vs. forest)
Divide region in 4453 blocks
Compute number of deforested and forest pixels per block
forestatrisk: 0

...10...20...30...40...50...60...70...80...90...100 - done
Draw blocks at random
Draw pixels at random in blocks
forestatrisk: 0...10...20...30...40...50...60...70...80...90...100 - done
Compute center of pixel coordinates
Compute number of 10 x 10 km spatial cells
... 2793 cells (49 x 57)
Identify cell number from XY coordinates
Make virtual raster with variables as raster bands
Extract raster values for selected pixels
forestatrisk: 0

RuntimeError: dist_river.tif, band 1: IReadBlock failed at X offset 43, Y offset 4: TIFFReadEncodedTile() failed.
May be caused by: TIFFReadEncodedTile() failed.
May be caused by: TIFFFillTile:Read error at row 12032, col 5632, tile 335; got 0 bytes, expected 261170

In [ ]:
from palmdef_risk.model.diagnostics import compute_vif
covariates = ["altitude", "slope", "dist_defor", "dist_edge", "dist_road",
              "dist_town", "dist_river", "gravity_resid"]
compute_vif(covariates, ctx.output_dir / "sample.csv",
            ctx.output_dir / "diagnostics" / "vif.json")

In [ ]:
from palmdef_risk.model.icar import fit_all
fit_all(ctx)

In [ ]:
from palmdef_risk.model.predict import predict_all
predict_all(ctx)

In [ ]:
from palmdef_risk.model.diagnostics import compute_residuals_all, compute_morans_i
residuals, coords = compute_residuals_all(ctx)
compute_morans_i(residuals, coords, ctx.output_dir / "diagnostics" / "moran.json")

In [ ]:
from palmdef_risk.model.sensitivity import run_gravity_sensitivity
run_gravity_sensitivity(ctx)

In [ ]:
from palmdef_risk.model.reports import export_all_diagnostics
export_all_diagnostics(ctx)